In [1]:
import sys
import logging
from pathlib import Path

import torch

PROJECT_ROOT = Path("..").resolve()
sys.path.insert(0, str(PROJECT_ROOT))

from utils.io_utils import load_fold
from utils.mlp_utils_lo import (
    set_seed,
    prepare_all_fold_tensors_lo,
    run_nested_random_search_lo,
    print_final_results_lo,
)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S",
)

logger = logging.getLogger(__name__)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
logger.info(f"Device: {device}")

GLOBAL_SEED = 42
set_seed(GLOBAL_SEED)

20:25:51 [INFO] Device: cuda


In [2]:
CFG = {
    "task":        "lo",
    "dataset":     "kdr",
    "fp_type":     "rdkit_desc",
    "n_bits":      1024,
    "outer_folds": [1, 2, 3],
    "inner_k":     2,
    "random_state": GLOBAL_SEED,
}

In [3]:
SEARCH_SPACE = {
    "n_layers":     [1, 2, 3],
    "n_nodes":      [64, 128, 256, 512, 1024],
    "r":            [0.5, 0.7, 1.0],
    "activation":   ["relu", "leaky_relu", "gelu", "silu"],  
    "dropout":      [0.0, 0.2, 0.3, 0.5],
    "batchnorm":    [True, False],
    "init":         ["kaiming", "xavier"],

    "lr":           [1e-4, 5e-4, 1e-3, 2e-3, 3e-3],
    "weight_decay": [0.0, 1e-5, 1e-4, 1e-3, 5e-3, 1e-2],
    "batch_size":   [64, 128, 256],
}

FIXED_HP = {
    "max_epochs": 100,
    "patience":   10,
    "grad_clip":  1.0,
}

N_ITER  = 150
N_SEEDS = 3

In [4]:
folds_data = {}

for fold_idx in CFG["outer_folds"]:
    train_df, test_df = load_fold(CFG["task"], CFG["dataset"], fold_idx)

    folds_data[fold_idx] = {"train": train_df, "test": test_df}

    logger.info(
        f"Fold {fold_idx} | "
        f"train={len(train_df)} "
        f"y_mean={train_df['value'].mean():.3f} "
        f"y_std={train_df['value'].std():.3f} | "
        f"test={len(test_df)} "
        f"n_clusters={test_df['cluster'].nunique()}"
    )

folds_tensors = prepare_all_fold_tensors_lo(
    cfg=CFG,
    folds_data=folds_data,
    logger=logger,
)

20:25:52 [INFO] Loaded lo/kdr fold 1: train=500, test=437
20:25:52 [INFO] Fold 1 | train=500 y_mean=6.769 y_std=0.894 | test=437 n_clusters=54
20:25:52 [INFO] Loaded lo/kdr fold 2: train=500, test=520
20:25:52 [INFO] Fold 2 | train=500 y_mean=6.774 y_std=0.856 | test=520 n_clusters=60
20:25:52 [INFO] Loaded lo/kdr fold 3: train=500, test=417
20:25:52 [INFO] Fold 3 | train=500 y_mean=6.739 y_std=0.865 | test=417 n_clusters=54
20:25:52 [INFO] Loading fingerprints from cache: /home/f.capria/drug-discovery-lohi/features/lo/kdr/rdkit_desc_train_1.npz
20:25:52 [INFO] Loading fingerprints from cache: /home/f.capria/drug-discovery-lohi/features/lo/kdr/rdkit_desc_test_1.npz
20:25:52 [INFO] Fold 1 | X_train: (500, 217), X_test: (437, 217) | scale_features=True | y_mean=6.769 y_std=0.893
20:25:52 [INFO] Loading fingerprints from cache: /home/f.capria/drug-discovery-lohi/features/lo/kdr/rdkit_desc_train_2.npz
20:25:52 [INFO] Loading fingerprints from cache: /home/f.capria/drug-discovery-lohi/featu

In [5]:
# ── Block 4 ── Nested CV random search ─────────────────────────────────────

logger.info(f"Estimated time: ~{N_ITER * 6 * len(CFG['outer_folds']) / 60:.0f} min")

fold_results = run_nested_random_search_lo(
    cfg=CFG,
    folds_tensors=folds_tensors,
    folds_data=folds_data,
    search_space=SEARCH_SPACE,
    fixed_hp=FIXED_HP,
    n_iter=N_ITER,
    n_seeds=N_SEEDS,
    device=device,
    seed=GLOBAL_SEED,
    logger=logger,
)

print_final_results_lo(fold_results, title="MLP KDR Lo — rdkit")

20:25:52 [INFO] Estimated time: ~45 min
20:25:52 [INFO] 
OUTER FOLD 1
20:25:53 [INFO]   [1/150] inner MSE=1865.9276 (score=-1865.9276) (2s) | L=2 N=512 r=0.7 dropout=0.3 lr=3e-03
20:25:55 [INFO]   [2/150] inner MSE=4062.7287 (score=-4062.7287) (1s) | L=3 N=128 r=1.0 dropout=0.2 lr=1e-03
20:25:55 [INFO]   [3/150] inner MSE=11445.6447 (score=-11445.6447) (0s) | L=1 N=1024 r=0.7 dropout=0.0 lr=5e-04
20:25:56 [INFO]   [4/150] inner MSE=2418.6022 (score=-2418.6022) (1s) | L=2 N=64 r=0.7 dropout=0.3 lr=3e-03
20:25:56 [INFO]   [5/150] inner MSE=62.2449 (score=-62.2449) (0s) | L=2 N=256 r=1.0 dropout=0.5 lr=3e-03
20:25:58 [INFO]   [6/150] inner MSE=3133.9711 (score=-3133.9711) (2s) | L=1 N=64 r=0.7 dropout=0.5 lr=1e-04
20:26:00 [INFO]   [7/150] inner MSE=1274.0767 (score=-1274.0767) (2s) | L=3 N=128 r=1.0 dropout=0.3 lr=1e-03
20:26:03 [INFO]   [8/150] inner MSE=1387.3580 (score=-1387.3580) (2s) | L=2 N=64 r=0.7 dropout=0.5 lr=5e-04
20:26:04 [INFO]   [9/150] inner MSE=9592.1569 (score=-9592.156


MLP KDR Lo — rdkit
Fold 1: mean_spearman=0.1288
Fold 2: mean_spearman=0.0885
Fold 3: mean_spearman=0.1122

Aggregated metrics:
  mean_spearman_mean: 0.1098
  mean_spearman_std: 0.0165
  std_spearman_mean: 0.4481
  std_spearman_std: 0.0055
  mean_r2_mean: -1.0404
  mean_r2_std: 0.3082
  mean_mae_mean: 0.8971
  mean_mae_std: 0.0266
  n_clusters_mean: 56.0
  n_clusters_std: 2.8284

Best hyperparameters per fold:
Fold 1: L=1 N=512 r=0.5 act=leaky_relu dropout=0.5 bn=False init=xavier lr=1e-03 wd=1e-03 bs=64
Fold 2: L=2 N=1024 r=0.5 act=relu dropout=0.3 bn=False init=kaiming lr=2e-03 wd=1e-02 bs=64
Fold 3: L=2 N=1024 r=0.5 act=relu dropout=0.5 bn=False init=xavier lr=2e-03 wd=0e+00 bs=64


{'mean_spearman_mean': np.float64(0.1098),
 'mean_spearman_std': np.float64(0.0165),
 'std_spearman_mean': np.float64(0.4481),
 'std_spearman_std': np.float64(0.0055),
 'mean_r2_mean': np.float64(-1.0404),
 'mean_r2_std': np.float64(0.3082),
 'mean_mae_mean': np.float64(0.8971),
 'mean_mae_std': np.float64(0.0266),
 'n_clusters_mean': np.float64(56.0),
 'n_clusters_std': np.float64(2.8284)}